In [32]:
import pandas as pd
import numpy as np
import importlib
import gc
import sys
import warnings
sys.path.append('./scripts')  
import preprocesamiento
import feature_engineering
import model_xgboost
import model_xgboost_keepsimple
importlib.reload(preprocesamiento)
importlib.reload(model_xgboost)
importlib.reload(model_xgboost_keepsimple)
importlib.reload(feature_engineering)
warnings.filterwarnings("ignore")

# Experimento 10: 
- XGBOOST
- Usando funcion entrenamiento: model_xgboost.semillerio_en_prediccion(train, test, version="v1")
- optuna = sqlite:///optuna_studies_v25.db
- Kaggle =  


##### Levantamos el dataset con target ya calculado

In [2]:
df = pd.read_csv("./datasets/periodo_x_producto_con_target.csv", sep=',', encoding='utf-8')
df.shape

(31362, 19)

In [3]:
df[df['target'].isna()][['product_id', 'periodo', 'tn', 'target']]
# 20034 , 201905

,product_id,periodo,tn,target
34,20001,201911,1397.37231,NaN
35,20001,201912,1504.68856,NaN
70,20002,201911,1423.57739,NaN
71,20002,201912,1087.30855,NaN
106,20003,201911,948.29393,NaN
...,...,...,...,...
31344,21274,201708,0.00867,NaN
31353,21276,201911,0.03341,NaN
31354,21276,201912,0.00892,NaN
31360,21281,201707,0.00000,NaN


In [26]:
columnas_baseline = df.columns.tolist()
columnas_baseline

['product_id',
 'periodo',
 'nacimiento_producto',
 'muerte_producto',
 'mes_n',
 'total_meses',
 'producto_nuevo',
 'ciclo_de_vida_inicial',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'sku_size',
 'stock_final',
 'tn',
 'plan_precios_cuidados',
 'cust_request_qty',
 'cust_request_tn',
 'target']

##### Preprocesamiento a la minima expresión :)

In [4]:
df = feature_engineering.create_category_features_cat1(df)
df = feature_engineering.create_category_features_cat2(df)
df = feature_engineering.create_category_features_cat3(df)

##### aplicamos OHE
# df = preprocesamiento.aplicarOHE(df)
df.shape

(31362, 28)

### Feature Engineering

##### Neural Prophet

In [5]:
neural_prophet_fe = pd.read_csv("./datasets/features_neuralprophet_completo.csv", sep=',', encoding='utf-8')
neural_prophet_fe['ds'] = pd.to_datetime(neural_prophet_fe['ds'], errors='coerce')
# Versión alternativa más robusta:
neural_prophet_fe['periodo'] = neural_prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
neural_prophet_fe = neural_prophet_fe[['periodo', 'product_id', 'trend', "season_yearly", "season_monthly"]]
df = df.merge(neural_prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 31)

##### Prophet

In [6]:
prophet_fe = pd.read_csv("./datasets/prophet_features_tn_zscore.csv", sep=',', encoding='utf-8')
prophet_fe['ds'] = pd.to_datetime(prophet_fe['ds'], errors='coerce')
prophet_fe['periodo'] = prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
prophet_fe = prophet_fe[['periodo', 'product_id', 'trend_add', "yearly_add", "additive_terms", 'trend_mult', 'yearly_mult', 'multiplicative_terms']]
df = df.merge(prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 37)

##### FE Moviles

In [7]:
df = feature_engineering.get_lags(df, "tn", 201912)
df = feature_engineering.get_delta_lags(df, "tn", 24)
df = feature_engineering.get_rolling_means(df, "tn", 201912)
df = feature_engineering.get_rolling_stds(df, "tn", 201912)
df = feature_engineering.get_rolling_mins(df, "tn", 201912)
df = feature_engineering.get_rolling_maxs(df, "tn", 201912)
df = feature_engineering.get_rolling_medians(df, "tn", 201912)
df = feature_engineering.get_rolling_skewness(df, "tn", 201912)
df = feature_engineering.get_autocorrelaciones(df, "tn", 201912)
df.shape

(31362, 599)

In [8]:
df = feature_engineering.get_lags(df, "cust_request_qty", 201912)
df = feature_engineering.get_delta_lags(df, "cust_request_qty", 24)
df = feature_engineering.get_rolling_means(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_stds(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_mins(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_maxs(df, "cust_request_qty", 201912)

In [ ]:
df = feature_engineering.get_lags(df, "cust_request_qty", 201912)
df = feature_engineering.get_delta_lags(df, "cust_request_qty", 24)
df = feature_engineering.get_rolling_means(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_stds(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_mins(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_maxs(df, "cust_request_qty", 201912)
df.shape

(31362, 1500)

##### FE Diana

In [8]:
df = feature_engineering.calcular_diferencia_con_medias_moviles(df)
df = feature_engineering.calcular_ratios_con_medias_moviles(df)
df.shape

(31362, 635)

##### FE Moviles sobre otras variables

In [56]:
# #  stock final
# df = feature_engineering.get_lagsEspecificos(df, col='stock_final_zscore')
# df = feature_engineering.get_delta_lags_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_means_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_stds_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_medians_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_mins_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='stock_final_zscore')

#  cust_request_qty
# df = feature_engineering.get_lagsEspecificos(df, col='cust_request_qty')
# df = feature_engineering.get_delta_lags_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_means_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_stds_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_mins_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_medians_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='cust_request_qty')

##### FE Calendario

In [9]:
df = feature_engineering.generar_ids(df)
df = feature_engineering.get_componentesTemporales(df)
df = feature_engineering.get_anomaliasPoliticas(df)
# df = feature_engineering.descomposicion_serie_temporal(df, col='tn')
df.shape

(31362, 660)

##### FE sobre FE

In [10]:
df = feature_engineering.chatGPT_features_serie(df, "tn")
df = feature_engineering.mes_con_feriado(df)
df.shape

(31362, 689)

##### Variables Exogenas

In [11]:
df = feature_engineering.get_dolar(df)
df = feature_engineering.get_IPC(df)
df['ipc'] = df['ipc'].str.replace(',', '.').astype(float)
df['dolar'] = df['dolar'].str.replace(',', '.').astype(float)
# df.drop(columns=['ds'], inplace=True)
#df.fillna(0, inplace=True) ##### EXPERIMENTAR
df = feature_engineering.correlacion_exogenas(df)
df = feature_engineering.get_mes_receso_escolar(df)
df.shape

(31362, 692)

##### Nuevas FE

In [12]:
df = feature_engineering.create_ratio_features(df)
df = feature_engineering.enhance_lifecycle_features(df)
# df = feature_engineering.create_category_features(df)
df = feature_engineering.create_regime_features(df)
df = feature_engineering.create_nonlinear_trends(df)
df = feature_engineering.create_temporal_interactions(df)
df = feature_engineering.create_asymmetric_window_features(df)
# df = feature_engineering.recomendaciones_deepseek(df)
df = feature_engineering.get_nuevas_features(df)
df.shape

(31362, 727)

##### Ceros

In [ ]:
df = feature_engineering.agregar_ceros_consecutivos_atras(df, col_tn='tn_original')
df = feature_engineering.agregar_no_ceros_consecutivos_atras(df, col_tn='tn_original')
df = feature_engineering.agregar_ceros_ultimos_n_meses(df, ventanas=[1,2,3,4,5,6,12], col_tn='tn_original')
df = feature_engineering.agregar_min_max_ult_n(df, n_list=(1,2,3,4,5,6,12), col_tn='tn_original')
df.shape

##### Elimino aquellas que no sirven

In [13]:
importantes = pd.read_csv("./feature_importance/exp04_3.csv", sep=',', encoding='utf-8')
no_importantes = importantes[importantes['importance'] <= 100]
no_importantes = no_importantes[~no_importantes['feature'].isin(columnas_baseline)]
no_importantes

,feature,importance
67,tn_vs_prev_year,100
68,tn_delta_lag5_lag8,99
69,tn_rolling_min_4,96
70,tn_lag_5,95
71,tn_rolling_std_6,95
...,...,...
677,dia_del_year,0
678,cat2_TE,0
680,cat2_PIEL1,0
681,cat2_OTROS,0


In [14]:
cols_a_eliminar = no_importantes.feature.unique()
print(f"Antes de eliminar: {df.shape[1]} columnas")
df = df.drop(columns=cols_a_eliminar, errors='ignore')
print(f"Después de eliminar: {df.shape[1]} columnas")

Antes de eliminar: 1144 columnas
Después de eliminar: 684 columnas


Eliminar object/categorical columnas

In [15]:
df = df.select_dtypes(exclude=['datetime', 'datetime64', 'object'])

Train Test Split

In [13]:
training = [
    201701, 201702, 201703, 201704, 201705, 201706, 201707, 201708, 201709,
    201710, 201711, 201712, 201801, 201802, 201803, 201804, 201805,
    201806, 201807, 201808, 201809, 201810, 201811, 201812,
    201901, 201902, 201903, 201904, 201905, 201906, 201907, 201908
]

validation = [
    201909
]


testing = [
    201910
]

prediction = [
    201912  
]

In [14]:
df_train = df[df['periodo'].isin(training)].copy()
df_val = df[df['periodo'].isin(validation)].copy()
df_test = df[df['periodo'].isin(testing)].copy()
df_pred = df[df['periodo'].isin(prediction)].copy()

gc.collect()

0

Entrenamiento

In [59]:
# Hay casos como este donde el producto muere en 2019006, por lo tanto tienen los dos ultimos target vacios.
# df_train[df_train['product_id']==20034][['product_id', 'periodo', 'tn', 'target']]
# 20034 , 201905

In [18]:
df_train['target'].fillna(0, inplace=True)
df_val['target'].fillna(0, inplace=True)
df_test['target'].fillna(0, inplace=True)

In [34]:
model_xgboost_keepsimple.optimizar_con_optuna_sin5FCV_con_semillerio_db(df_train=df_train, df_val=df_val, version="v26", n_trials=500)


Para visualizar los resultados en tiempo real:
1. Abre otra terminal y ejecuta:
   optuna-dashboard sqlite:///optuna_studies_v26.db
2. Abre en tu navegador: http://127.0.0.1:8080/


[I 2025-07-16 09:03:17,060] A new study created in RDB with name: xgboost_optimization_v26


⏳ Ejecutando Trial #0
[0]	validation_0-rmse:101.86148
[1]	validation_0-rmse:98.63045
[2]	validation_0-rmse:95.86212
[3]	validation_0-rmse:93.02808
[4]	validation_0-rmse:90.16587
[5]	validation_0-rmse:87.30848
[6]	validation_0-rmse:84.88291
[7]	validation_0-rmse:81.95609
[8]	validation_0-rmse:79.71684
[9]	validation_0-rmse:77.07052
[10]	validation_0-rmse:74.47437
[11]	validation_0-rmse:72.14805
[12]	validation_0-rmse:70.17291
[13]	validation_0-rmse:68.06698
[14]	validation_0-rmse:66.30857
[15]	validation_0-rmse:64.76947
[16]	validation_0-rmse:63.37569
[17]	validation_0-rmse:62.00684
[18]	validation_0-rmse:60.41785
[19]	validation_0-rmse:58.91747
[20]	validation_0-rmse:57.21977
[21]	validation_0-rmse:55.83287
[22]	validation_0-rmse:54.41938
[23]	validation_0-rmse:53.18278
[24]	validation_0-rmse:51.96990
[25]	validation_0-rmse:50.67443
[26]	validation_0-rmse:49.79511
[27]	validation_0-rmse:48.97135
[28]	validation_0-rmse:48.08121
[29]	validation_0-rmse:47.02765
[30]	validation_0-rmse:45.9

[W 2025-07-16 09:06:59,095] Trial 0 failed with parameters: {'learning_rate': 0.03574712922600244, 'n_estimators': 956, 'max_depth': 8, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'lambda': 3.3323645788192616e-08, 'alpha': 0.6245760287469893, 'max_bin': 348} because of the following error: XGBoostError('value -1 for Parameter verbosity exceed bound [0,3]\nverbosity: Flag to print out detailed breakdown of runtime.').
Traceback (most recent call last):
  File "c:\Users\Usuario\.conda\envs\py311lab3\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "c:\Users\Usuario\Documents\Universidad\austral\2025\Lab3\Lab3-MCD\notebooks\entregable\./scripts\model_xgboost_keepsimple.py", line 163, in objective
    model.fit(
  File "c:\Users\Usuario\.conda\envs\py311lab3\Lib\site-packages\xgboost\core.py", line 729, in inner_f
    return func(**kwargs)
    

XGBoostError: value -1 for Parameter verbosity exceed bound [0,3]
verbosity: Flag to print out detailed breakdown of runtime.

Prediccion

In [38]:
df_future = model_lgb.semillerio_en_prediccion_con_pesos(train, test, version="v1")

In [39]:
df_future

,periodo,product_id,target,pred
30476,201912,20001,0.0,1490.523819
30477,201912,20002,0.0,1176.601177
30478,201912,20003,0.0,760.653373
30479,201912,20004,0.0,713.971133
30480,201912,20005,0.0,610.816894
...,...,...,...,...
31357,201912,21265,0.0,-8.235051
31358,201912,21266,0.0,-8.913198
31359,201912,21267,0.0,-5.841841
31360,201912,21271,0.0,-1.394455


In [ ]:
productos_ok = pd.read_csv("https://storage.googleapis.com/open-courses/austral2025-af91/labo3v/product_id_apredecir201912.txt", sep="\t")
df_future = df_future[df_future['periodo'] == 201912]
df_future = df_future[df_future['product_id'].isin(productos_ok['product_id'].unique())]


,periodo,product_id,target,pred
30476,201912,20001,0.0,1490.523819
30477,201912,20002,0.0,1176.601177
30478,201912,20003,0.0,760.653373
30479,201912,20004,0.0,713.971133
30480,201912,20005,0.0,610.816894
...,...,...,...,...
31355,201912,21263,0.0,-6.386380
31357,201912,21265,0.0,-8.235051
31358,201912,21266,0.0,-8.913198
31359,201912,21267,0.0,-5.841841


In [ ]:
linear_regression = pd.read_csv("./outputs/predicciones_regresion_lineal_v2.csv", sep=',', encoding='utf-8')
df_future = df_future[['product_id', 'pred']]
df_future = df_future.merge(linear_regression, on='product_id', how='left')
df_future.loc[df_future['pred'] < 0, 'pred'] = df_future['tn']
# Definir el rango de product_ids
mask = (df_future['product_id'] >= 20001) & (df_future['product_id'] <= 20005)

# Asignar el valor de tn a pred para esos productos
df_future.loc[mask, 'pred'] = df_future.loc[mask, 'tn']
df_future



,product_id,pred,tn
0,20001,1232.764336,1232.764336
1,20002,1135.499014,1135.499014
2,20003,683.869748,683.869748
3,20004,551.448542,551.448542
4,20005,546.346434,546.346434
...,...,...,...
775,21263,0.249044,0.249044
776,21265,0.072234,0.072234
777,21266,0.076151,0.076151
778,21267,0.075869,0.075869


In [45]:
df_future.drop(columns=['tn'], inplace=True)
df_future.rename(columns={'pred': 'tn'}, inplace=True)
df_future.to_csv("./outputs/predicciones_exp_07_lgb_v1.csv", index=False, sep=',')

Feature importance

In [78]:
model_lgb.feature_importance(df, models, "05")

In [79]:
importances = model_lgb.feature_importance_promedio(models, df.drop(columns=['target']))
importances.sort_values(by='importance', ascending=False, inplace=True) 
importances

,feature,importance
16,yearly_add,0.025628
0,product_id,0.017944
1084,tn_seasonal_naive,0.014613
1119,naive_forecast_error,0.014388
1080,tn_ytd_sum,0.012998
...,...,...
1053,stock_final_rolling_max_18,0.000000
1052,stock_final_rolling_max_17,0.000000
1051,stock_final_rolling_max_16,0.000000
1050,stock_final_rolling_max_15,0.000000
